# Comparator Table

This notebook reports reference models that help interpret the primary SRM composites.

**Comparator roles**

| Comparator | Nature | Input | Output | Why included |
|---|---|---|---|---|
| LDA | Linear Discriminant Analysis visit-separation direction | Imaging visit rows | Projection score | Tests whether visits can be separated by imaging patterns. |
| Regression reference | ElasticNet clinical target model | Imaging visit rows | Predicted clinical score | Tests whether imaging predicts clinical scales used as benchmarks. |

**Interpretation warning:** LDA Fisher-criterion separation and paired SRM-based progression `d_z` are related but not directly interchangeable.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("Could not find repo root")

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import DEFAULT_CONFIG, set_global_seeds
from src.data.trackfa_pairs import infer_trackfa_feature_groups, trackfa_pairs_to_long
from src.eval.cv import interaction_loocv, lda_loocv, lda_nested_loocv, tune_and_run_regression_loocv
from src.eval.intervals import adjacent_pair_interval_effect_summary, annual_tuning_diagnostics
from src.eval.metrics import bootstrap_ci_d, clinical_change_effect_sizes, reference_effect_sizes
from src.eval.optimization import optimization_log, optimization_row, save_optimization_log
from src.reporting.tuning_review import plot_tuning_review, tuning_recommendation, tuning_verification_summary
from src.features.selection import feature_stability_report
from src.models.srm_global import srm_global_loocv

set_global_seeds(DEFAULT_CONFIG.random_state)
pairs_path = REPO_ROOT / "data" / "processed" / "trackfa_pairs_drop3poms.csv"
if not pairs_path.exists():
    raise FileNotFoundError(f"Required modelling dataset not found: {pairs_path}")
pairs_df = pd.read_csv(pairs_path)
long_df = trackfa_pairs_to_long(pairs_df)
groups = infer_trackfa_feature_groups(pairs_df)
imaging_cols = [c for c in groups.all_neuroimaging if c in long_df.columns]
subject_col = "pair_id"  # progression interval id, e.g. AAN001_V1V2
split_group_col = "subject"  # participant id; keeps V1V2 and V2V3 in the same fold
selection_method = "none"
selection_k = 8
LDA_CV_N_SPLITS = DEFAULT_CONFIG.cv_n_splits  # Subject-level grouped CV for annual-consistency tuning.
REGRESSION_CV_N_SPLITS = DEFAULT_CONFIG.cv_n_splits
N_BOOT = 300
RANDOM_SEED = DEFAULT_CONFIG.random_state
RIDGE_GRID = [1e-8, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0]
COVARIANCE_SHRINKAGE_GRID = [0.75, 1.0]
LDA_Z_CLIP_GRID = [None, 4.0]
ELASTICNET_L1_RATIO_GRID = [0.2, 0.5, 0.8, 1.0]
REGRESSION_Z_CLIP_GRID = [None, 4.0]
LDA_SHRINK_GRID = ["auto", 1e-8, 0.1, 1.0, 10.0]
REGRESSION_PARAM_SELECTION_METRIC = "annual_mean_dz"
REGRESSION_MODEL_KINDS = ["elasticnet"]
optimization_rows = []
print(f"Loaded {pairs_path.name}: {long_df.shape[0]} visit rows, {len(imaging_cols)} imaging features")
print({"selection_method": selection_method, "selection_k": selection_k, "lda_cv_n_splits": LDA_CV_N_SPLITS,
    "regression_cv_n_splits": REGRESSION_CV_N_SPLITS, "regression_param_selection_metric": REGRESSION_PARAM_SELECTION_METRIC})
def benchmark_table(model_name: str, d_score: float, ci_low: float, ci_high: float) -> pd.DataFrame:
    imaging_ref = reference_effect_sizes(
        long_df,
        imaging_cols=imaging_cols,
        scale_cols=(),
        subject_col=subject_col,
        visit_col="visit",
    )
    clinical_ref = clinical_change_effect_sizes(
        pairs_df,
        scale_cols=("FARS", "SARA"),
        pair_types=("V1V2", "V2V3"),
    )
    rows = [{"feature": model_name, "kind": "model", "d": d_score, "ci_low": ci_low, "ci_high": ci_high, "source_delta_col": np.nan, "pair_types": np.nan}]
    for scale in ("FARS", "SARA"):
        hit = clinical_ref[(clinical_ref["kind"] == "scale") & (clinical_ref["feature"] == scale)].head(1)
        if len(hit):
            r = hit.iloc[0].to_dict()
            rows.append({"feature": r["feature"], "kind": r["kind"], "d": r["d"], "ci_low": np.nan, "ci_high": np.nan, "source_delta_col": r.get("source_delta_col", np.nan), "pair_types": r.get("pair_types", np.nan)})
    top_img = imaging_ref[imaging_ref["kind"] == "imaging"].head(1)
    if len(top_img):
        r = top_img.iloc[0].to_dict()
        rows.append({"feature": r["feature"], "kind": r["kind"], "d": r["d"], "ci_low": np.nan, "ci_high": np.nan, "source_delta_col": r.get("source_delta_col", np.nan), "pair_types": r.get("pair_types", np.nan)})
    return pd.DataFrame(rows)


Loaded trackfa_pairs_drop3poms.csv: 414 visit rows, 146 imaging features
{'selection_method': 'none', 'selection_k': 8, 'lda_cv_n_splits': 5, 'regression_cv_n_splits': 5, 'regression_param_selection_metric': 'annual_mean_dz'}


## 1. LDA Visit-Separation Comparator

Linear Discriminant Analysis (LDA) finds a direction that separates visit labels. It is useful as a comparator, but the project target remains paired progression change on held-out participant groups.


In [2]:
import time

lda_trials = []
for z_clip in LDA_Z_CLIP_GRID:
    for covariance_shrinkage in COVARIANCE_SHRINKAGE_GRID:
        for shrink in LDA_SHRINK_GRID:
            start = time.time()
            res = lda_loocv(
                long_df,
                imaging_cols,
                subject_col=subject_col,
                visit_col="visit",
                selection_method=selection_method,
                k=selection_k,
                cv_n_splits=LDA_CV_N_SPLITS,
                random_seed=RANDOM_SEED,
                split_group_col=split_group_col,
                shrink=shrink,
                covariance_shrinkage=covariance_shrinkage,
                z_clip=z_clip,
                compute_ci=False,
            )
            interval_summary = adjacent_pair_interval_effect_summary(
                res["oof_df"],
                pair_col=subject_col,
                visit_col="visit",
                score_col="score",
                n_boot=N_BOOT,
                seed=RANDOM_SEED,
            )
            res = {**res, **annual_tuning_diagnostics(interval_summary)}
            row = optimization_row(
                model="LDA exploratory",
                params={
                    "shrink": shrink,
                    "covariance_shrinkage": covariance_shrinkage,
                    "z_clip": z_clip,
                    "selection_method": selection_method,
                    "regularization": "ridge_plus_covariance_shrinkage_plus_optional_z_clip",
                },
                result=res,
                runtime_sec=time.time() - start,
                notes="exploratory grid reported with annual mean d_z and V1->V2/V2->V3 consistency diagnostics",
            )
            lda_trials.append((res, row))
            optimization_rows.append(row)

lda_optimization_df = optimization_log([row for _, row in lda_trials], sort_by="mean_validation_annual_dz")
display(lda_optimization_df)

lda_review = tuning_recommendation(lda_optimization_df)
print("Numerically best LDA configuration")
display(pd.DataFrame([lda_review["raw_best"]]))
print("One-SE / near-optimal LDA candidate set")
display(lda_review["near_optimal"])
print("Recommended LDA configuration by implemented hierarchy")
display(pd.DataFrame([lda_review["recommended"]]))
print(lda_review["summary"])
fig = plot_tuning_review(lda_review["review_table"], title="LDA Annual Tuning Review")
if fig is not None:
    display(fig)
print("Human-verification summary")
display(tuning_verification_summary(lda_review))

lda_nested_candidates = [
    {
        "shrink": shrink,
        "covariance_shrinkage": covariance_shrinkage,
        "z_clip": z_clip,
        "selection_method": selection_method,
        "k": selection_k,
    }
    for z_clip in LDA_Z_CLIP_GRID + [3.0]
    for covariance_shrinkage in COVARIANCE_SHRINKAGE_GRID
    for shrink in LDA_SHRINK_GRID
]
start = time.time()
lda_res = lda_nested_loocv(
    long_df,
    imaging_cols,
    subject_col=subject_col,
    visit_col="visit",
    candidates=lda_nested_candidates,
    cv_n_splits=LDA_CV_N_SPLITS,
    inner_folds=5,
    random_seed=RANDOM_SEED,
    split_group_col=split_group_col,
    compute_ci=True,
    tuning_metric="annual_mean_dz",
)
lda_interval_summary = adjacent_pair_interval_effect_summary(
    lda_res["oof_df"],
    pair_col=subject_col,
    visit_col="visit",
    score_col="score",
    n_boot=N_BOOT,
    seed=RANDOM_SEED,
)
lda_res = {**lda_res, **annual_tuning_diagnostics(lda_interval_summary)}
optimization_rows.append(optimization_row(
    model="LDA nested",
    params={"candidate_count": len(lda_nested_candidates), "inner_folds": 5, "tuning": "train-fold inner grouped CV"},
    result=lda_res,
    runtime_sec=time.time() - start,
    notes="nested estimate with annual interval diagnostics; tune/report mean annual d_z and interval gap",
))
display(lda_res["chosen_params_df"].head())
display(lda_interval_summary)
pd.DataFrame([{
    "model": "LDA nested",
    "selection_method": selection_method,
    "candidate_count": len(lda_nested_candidates),
    "dz_v1_v2": lda_res["dz_v1_v2"],
    "dz_v2_v3": lda_res["dz_v2_v3"],
    "mean_annual_d_z": lda_res["mean_validation_annual_dz"],
    "annual_interval_gap": lda_res["annual_interval_gap"],
    "pooled_pair_d_z_reference": lda_res["d_score"],
    "ci_low": lda_res["d_ci_low"],
    "ci_high": lda_res["d_ci_high"],
    "n_subject_pairs": lda_res["n_subjects"],
}])


,model,feature_pool,objective,d_score,dz_v1_v2,dz_v2_v3,mean_validation_annual_dz,annual_interval_gap,p_progression,se_validation_dz,...,cv_n_splits,split_group_col,n_split_groups,runtime_sec,notes,param_shrink,param_covariance_shrinkage,param_z_clip,param_selection_method,param_regularization
0,LDA exploratory,all_imaging,d_score,0.455172,0.596085,0.323636,0.459861,0.272449,0.688131,NaN,...,5,subject,117,0.104383,exploratory grid reported with annual mean d_z...,10.0,1.00,4.0,none,ridge_plus_covariance_shrinkage_plus_optional_...
1,LDA exploratory,all_imaging,d_score,0.454680,0.596186,0.322520,0.459353,0.273666,0.688131,NaN,...,5,subject,117,0.116592,exploratory grid reported with annual mean d_z...,1.0,1.00,4.0,none,ridge_plus_covariance_shrinkage_plus_optional_...
2,LDA exploratory,all_imaging,d_score,0.454434,0.596670,0.321524,0.459097,0.275146,0.688131,NaN,...,5,subject,117,0.162328,exploratory grid reported with annual mean d_z...,0.1,1.00,4.0,none,ridge_plus_covariance_shrinkage_plus_optional_...
3,LDA exploratory,all_imaging,d_score,0.454433,0.596854,0.321336,0.459095,0.275518,0.688131,NaN,...,5,subject,117,0.090414,exploratory grid reported with annual mean d_z...,0.0,1.00,4.0,none,ridge_plus_covariance_shrinkage_plus_optional_...
4,LDA exploratory,all_imaging,d_score,0.454433,0.596852,0.321338,0.459095,0.275514,0.688131,NaN,...,5,subject,117,0.104438,exploratory grid reported with annual mean d_z...,auto,1.00,4.0,none,ridge_plus_covariance_shrinkage_plus_optional_...
5,LDA exploratory,all_imaging,d_score,0.454212,0.589232,0.328331,0.458782,0.260901,0.693603,NaN,...,5,subject,117,0.127217,exploratory grid reported with annual mean d_z...,10.0,1.00,NaN,none,ridge_plus_covariance_shrinkage_plus_optional_...
6,LDA exploratory,all_imaging,d_score,0.454116,0.589107,0.328255,0.458681,0.260852,0.693603,NaN,...,5,subject,117,0.108214,exploratory grid reported with annual mean d_z...,1.0,1.00,NaN,none,ridge_plus_covariance_shrinkage_plus_optional_...
7,LDA exploratory,all_imaging,d_score,0.454020,0.588981,0.328179,0.458580,0.260802,0.693603,NaN,...,5,subject,117,0.113198,exploratory grid reported with annual mean d_z...,0.1,1.00,NaN,none,ridge_plus_covariance_shrinkage_plus_optional_...
8,LDA exploratory,all_imaging,d_score,0.453999,0.588953,0.328162,0.458558,0.260791,0.693603,NaN,...,5,subject,117,0.126905,exploratory grid reported with annual mean d_z...,auto,1.00,NaN,none,ridge_plus_covariance_shrinkage_plus_optional_...
9,LDA exploratory,all_imaging,d_score,0.453999,0.588953,0.328162,0.458557,0.260791,0.693603,NaN,...,5,subject,117,0.085992,exploratory grid reported with annual mean d_z...,0.0,1.00,NaN,none,ridge_plus_covariance_shrinkage_plus_optional_...


Numerically best LDA configuration


,param_shrink,param_covariance_shrinkage,param_z_clip,param_selection_method,param_regularization,dz_v1_v2,dz_v2_v3,mean_validation_annual_dz,annual_interval_gap,p_progression,feature_count,jaccard_stability,sign_stability,coefficient_sign_stability,se_validation_dz,raw_rank
0,10.0,1.0,4.0,none,ridge_plus_covariance_shrinkage_plus_optional_...,0.596085,0.323636,0.459861,0.272449,0.688131,NaN,NaN,NaN,NaN,NaN,1.0


One-SE / near-optimal LDA candidate set


,param_shrink,param_covariance_shrinkage,param_z_clip,param_selection_method,param_regularization,dz_v1_v2,dz_v2_v3,mean_validation_annual_dz,annual_interval_gap,p_progression,feature_count,jaccard_stability,sign_stability,coefficient_sign_stability,se_validation_dz,raw_rank
0,10.0,1.0,4.0,none,ridge_plus_covariance_shrinkage_plus_optional_...,0.596085,0.323636,0.459861,0.272449,0.688131,NaN,NaN,NaN,NaN,NaN,1.0


...

Human-verification summary


,item,value
0,Best raw-performance parameters,"{'shrink': 10.0, 'covariance_shrinkage': 1.0, ..."
1,Recommended parameters,"{'shrink': 10.0, 'covariance_shrinkage': 1.0, ..."
2,Difference in performance,0.0
3,Reason for recommendation,Recommended candidate is also the raw best by ...
4,Any instability/warning,No automatic warning.


,outer_fold,inner_d_score,inner_tuning_metric,inner_dz_v1_v2,inner_dz_v2_v3,inner_annual_interval_gap,inner_p_progression,n_features,shrink,covariance_shrinkage,z_clip,selection_method,k
0,1,0.391131,annual_mean_dz,0.549668,0.232595,0.317074,0.669872,146,10.0,0.75,4.0,none,8
1,2,0.395407,annual_mean_dz,0.619857,0.170958,0.448899,0.667619,146,10.0,0.75,NaN,none,8
2,3,0.487123,annual_mean_dz,0.627371,0.346876,0.280495,0.673395,146,10.0,0.75,NaN,none,8
3,4,0.398929,annual_mean_dz,0.556442,0.241417,0.315025,0.679310,146,10.0,0.75,NaN,none,8
4,5,0.400069,annual_mean_dz,0.575478,0.224659,0.350819,0.673457,146,10.0,0.75,NaN,none,8


,interval,n_pairs,mean_change,sd_change,d_z,d_z_ci_low,d_z_ci_high,p_delta_positive
0,V1->V2,108,0.030684,0.057621,0.532507,0.319137,0.791469,0.731481
1,V2->V3,99,0.017912,0.062386,0.287113,0.081873,0.502532,0.626263


,model,selection_method,candidate_count,dz_v1_v2,dz_v2_v3,mean_annual_d_z,annual_interval_gap,pooled_pair_d_z_reference,ci_low,ci_high,n_subject_pairs
0,LDA nested,none,30,0.532507,0.287113,0.40981,0.245394,0.408627,0.254527,0.5536,207


## 2. Regression Reference

Regression models predict clinical targets from imaging features. These outputs are references, not the primary biomarker objective, because the project aims to measure imaging progression rather than optimise clinical-score prediction.


In [3]:
import time

regression_rows = []
regression_results = {}
for target in [c for c in ["FARS", "SARA"] if c in long_df.columns]:
    for model_kind in REGRESSION_MODEL_KINDS:
        for z_clip in REGRESSION_Z_CLIP_GRID:
            start = time.time()
            res = tune_and_run_regression_loocv(
                long_df,
                imaging_cols,
                target_col=target,
                subject_col=subject_col,
                model_kind=model_kind,
                selection_method=selection_method,
                k=selection_k,
                visit_col="visit",
                cv_n_splits=REGRESSION_CV_N_SPLITS,
                param_selection_metric=REGRESSION_PARAM_SELECTION_METRIC,
                z_clip=z_clip,
                split_group_col=split_group_col,
            )
            interval_summary = adjacent_pair_interval_effect_summary(
                res["oof_df"],
                pair_col=subject_col,
                visit_col="visit",
                score_col="pred",
                n_boot=N_BOOT,
                seed=RANDOM_SEED,
            )
            annual_diag = annual_tuning_diagnostics(interval_summary)
            res = {**res, **annual_diag}
            regression_results[(target, model_kind, z_clip)] = res
            params = {
                "target": target,
                "model_kind": model_kind,
                "selection_method": selection_method,
                "param_selection_metric": REGRESSION_PARAM_SELECTION_METRIC,
                "z_clip": z_clip,
            }
            optimization_rows.append(optimization_row(
                model=f"Regression {model_kind}",
                params=params,
                result=res,
                runtime_sec=time.time() - start,
                notes="inner hyperparameters reported with annual mean d_z and V1->V2/V2->V3 consistency diagnostics",
            ))
            regression_rows.append({
                "target": target,
                "model": model_kind,
                "selection_method": selection_method,
                "param_selection_metric": REGRESSION_PARAM_SELECTION_METRIC,
                "z_clip": z_clip,
                "rmse": res["rmse"],
                "r2": res["r2"],
                "dz_v1_v2": res["dz_v1_v2"],
                "dz_v2_v3": res["dz_v2_v3"],
                "mean_annual_d_z": res["mean_validation_annual_dz"],
                "annual_interval_gap": res["annual_interval_gap"],
                "p_progression": res["p_progression"],
                "pooled_pair_d_z_reference": res["d_score"],
                "ci_low": res["d_ci_low"],
                "ci_high": res["d_ci_high"],
                "n_subject_pairs": res["n_subjects"],
            })
regression_df = pd.DataFrame(regression_rows)
optimization_df = optimization_log(optimization_rows, sort_by="mean_validation_annual_dz")
log_path = save_optimization_log(optimization_df, REPO_ROOT / "results" / "comparator_optimization_log.csv")
print("Saved optimization log:", log_path)
display(regression_df.sort_values(["mean_annual_d_z", "annual_interval_gap"], ascending=[False, True]))
display(optimization_df)

for review_model, review_group in optimization_df.groupby("model", sort=False):
    model_review = tuning_recommendation(review_group)
    print(f"Tuning review: {review_model}")
    print("Numerically best configuration")
    display(pd.DataFrame([model_review["raw_best"]]))
    print("One-SE / near-optimal candidate set")
    display(model_review["near_optimal"])
    print("Recommended configuration by implemented hierarchy")
    display(pd.DataFrame([model_review["recommended"]]))
    print(model_review["summary"])
    fig = plot_tuning_review(model_review["review_table"], title=f"{review_model} Annual Tuning Review")
    if fig is not None:
        display(fig)
    print("Human-verification summary")
    display(tuning_verification_summary(model_review))


/Users/robertwang/Library/Python/3.8/lib/python/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.047e+03, tolerance: 3.466e+00
  model = cd_fast.enet_coordinate_descent(
/Users/robertwang/Library/Python/3.8/lib/python/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.943e+02, tolerance: 4.355e+00
  model = cd_fast.enet_coordinate_descent(


/Users/robertwang/Library/Python/3.8/lib/python/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.767e+02, tolerance: 4.695e+00
  model = cd_fast.enet_coordinate_descent(
/Users/robertwang/Library/Python/3.8/lib/python/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.417e+02, tolerance: 3.466e+00
  model = cd_fast.enet_coordinate_descent(


/Users/robertwang/Library/Python/3.8/lib/python/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.605e+02, tolerance: 4.355e+00
  model = cd_fast.enet_coordinate_descent(
/Users/robertwang/Library/Python/3.8/lib/python/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.283e+01, tolerance: 4.695e+00
  model = cd_fast.enet_coordinate_descent(


/Users/robertwang/Library/Python/3.8/lib/python/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.035e+01, tolerance: 3.466e+00
  model = cd_fast.enet_coordinate_descent(
/Users/robertwang/Library/Python/3.8/lib/python/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.439e+01, tolerance: 4.355e+00
  model = cd_fast.enet_coordinate_descent(


/Users/robertwang/Library/Python/3.8/lib/python/site-packages/sklearn/linear_model/_coordinate_descent.py:628: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.669e+01, tolerance: 4.695e+00
  model = cd_fast.enet_coordinate_descent(


...

,param_shrink,param_covariance_shrinkage,param_z_clip,param_selection_method,param_regularization,param_candidate_count,param_inner_folds,param_tuning,param_target,param_model_kind,...,annual_interval_gap,p_progression,feature_count,jaccard_stability,sign_stability,coefficient_sign_stability,se_validation_dz,raw_rank,directional_consistency,score_ranking_stability
14,NaN,NaN,NaN,NaN,NaN,30.0,5.0,train-fold inner grouped CV,NaN,NaN,...,0.245394,0.678872,inf,-inf,-inf,-inf,NaN,1.0,-inf,-inf


Recommended candidate is also the raw best by mean annual validation d_z; the consistency, progression-probability, simplicity, and stability tie-breakers did not select a different row.


<Figure size 1200x400 with 3 Axes>

Human-verification summary


,item,value
0,Best raw-performance parameters,"{'shrink': nan, 'covariance_shrinkage': nan, '..."
1,Recommended parameters,"{'shrink': nan, 'covariance_shrinkage': nan, '..."
2,Difference in performance,0.0
3,Reason for recommendation,Recommended candidate is also the raw best by ...
4,Any instability/warning,No automatic warning.


## 3. Clinical Benchmark Table

The final display compares the best comparator row with FARS, SARA, and the top single imaging feature using the same paired Cohen's `d_z` convention where applicable.


In [4]:
best_reg = regression_df.sort_values(["mean_annual_d_z", "annual_interval_gap"], ascending=[False, True]).head(1)
if len(best_reg):
    row = best_reg.iloc[0]
    model_name = f"Regression {row['model']} ({row['target']})"
    model_d = row["pooled_pair_d_z_reference"]
    model_lo = row["ci_low"]
    model_hi = row["ci_high"]
else:
    model_name, model_d, model_lo, model_hi = "Regression reference", np.nan, np.nan, np.nan
lda_table = pd.DataFrame([{"feature": "LDA nested", "kind": "model", "d": lda_res["d_score"], "ci_low": lda_res["d_ci_low"], "ci_high": lda_res["d_ci_high"]}])
reg_table = benchmark_table(model_name, model_d, model_lo, model_hi)
display(pd.concat([lda_table, reg_table], ignore_index=True))


,feature,kind,d,ci_low,ci_high,source_delta_col,pair_types
0,LDA nested,model,0.408627,0.254527,0.553600,NaN,NaN
1,Regression elasticnet (FARS),model,0.421733,0.290408,0.574993,NaN,NaN
2,FARS,scale,0.407427,NaN,NaN,delta_mfars_total,"V1V2,V2V3"
3,SARA,scale,0.405463,NaN,NaN,delta_sara_total,"V1V2,V2V3"
4,cerebellumFS,imaging,-0.667820,NaN,NaN,NaN,NaN
